# NVIDIA (CUDA) environment check

Run this inside an NVIDIA DevEnvironment to confirm what the image carries and that the
GPU can actually be used by the account the environment runs as.

The device count reports what the container runtime exposes. It can be correct while
every kernel launch fails — an image compiled for an older CUDA architecture than the
card still reports a device and then refuses to run on it — so it is printed rather than
asserted. The last cell is the one that proves access, and it is also the first cell to
touch a device.

Give it memory. The first kernel launch initializes the CUDA context and the libraries
behind it, so the last cell needs more memory than everything above it.

The last cell also reports the same matmul twice. NVIDIA's torch builds turn TF32 on by
default (`float32_matmul_precision` is `high`), so the default result is not the strict
FP32 one; comparing it against a float64 CPU reference and reading the gap as a fault
would be wrong.


In [ ]:
# The interpreter this kernel runs on. In the platform's NVIDIA images it is the
# system python, which is the one that carries torch.
from platform import python_version
print(f"Python Version: {python_version()}")


In [ ]:
import torch
print(f"PyTorch Version: {torch.__version__}")
# The CUDA version this build was compiled against — not the driver's, and not
# necessarily the one the image tag names.
print(f"Built for CUDA: {torch.version.cuda}")


In [ ]:
# The driver's own view of the card, independent of torch: the version the node's
# container runtime hands this container. An absent binary or a stalled driver is
# reported rather than raised, so the access check below still runs.
import subprocess
try:
    out = subprocess.run(["nvidia-smi", "--query-gpu=driver_version,name",
                          "--format=csv,noheader"], capture_output=True, text=True,
                         timeout=30)
    print(out.stdout.strip() or out.stderr.strip())
except FileNotFoundError:
    print("nvidia-smi not found: the container runtime did not inject it")
except subprocess.TimeoutExpired:
    print("nvidia-smi did not answer within 30s")

In [ ]:
# What the environment was configured to see. Reading it opens no device, which is
# why it can be right while access fails — see the last cell.
count = torch.cuda.device_count()
print(f"Total Physical GPUs: {count}")


In [ ]:
# The access check. `is_available()` needs the device to be openable by this account,
# which is what the cell above cannot show: a non-root environment can report the right
# count and then fail here. If it is False, the node's container runtime is not handing
# this account the device.
print(f"CUDA Available: {torch.cuda.is_available()}")

# The names, and a real operation. The line above already initialized the driver, so
# neither is the first touch; they follow it rather than sitting with the count, because
# a device that cannot be opened would otherwise raise before the access result printed.
print([torch.cuda.get_device_name(i) for i in range(count)])
major, minor = torch.cuda.get_device_capability(0)
print(f"Compute Capability: sm_{major}{minor}")

x = torch.randn(1024, 1024, device="cuda")
print(f"1024x1024 matmul ok, sum = {float((x @ x).sum())}")

# The same matmul at both precisions. NVIDIA's builds turn TF32 on, which is why a
# default-precision result differs from a strict FP32 one, but torch's own default is
# the strict one, so whether this kernel started with TF32 is reported rather than
# assumed. The setting is global to the kernel, so it is restored afterwards rather
# than left at whichever case ran last.
ref = x[:256, :].cpu().double() @ x[:, :256].cpu().double()
initial = torch.backends.cuda.matmul.allow_tf32
print(f"TF32 enabled before this cell: {initial}")
try:
    for label, tf32 in (("TF32 (enabled)", True), ("FP32", False)):
        torch.backends.cuda.matmul.allow_tf32 = tf32
        err = ((x @ x)[:256, :256].cpu().double() - ref).abs().max().item()
        print(f"{label}: max |gpu - cpu64| = {err:.2e}")
finally:
    torch.backends.cuda.matmul.allow_tf32 = initial